# 🧠 Úvod do strojového učení s PyTorch

**SPŠ Kladno — Výukový materiál**

Tento notebook vás provede základy strojového učení pomocí frameworku **PyTorch**. Postupně se naučíte pracovat s tensory, připravovat data, stavět neuronové sítě, trénovat je a ukládat hotové modely.

---

## Proč PyTorch?

| Vlastnost | Popis |
|---|---|
| 🐍 **Pythonic** | Syntaxe blízká čistému Pythonu, snadno čitelná |
| 🔬 **Věda & výzkum** | Nejpoužívanější framework v akademickém prostředí |
| ⚡ **Dynamický graf** | Výpočetní graf se buduje za běhu — snadné debugování |
| 🖥️ **GPU podpora** | Jednoduché přepínání CPU ↔ GPU |
| 📦 **Ekosystém** | torchvision, torchaudio, torchtext, HuggingFace… |

---

## Obsah

| # | Téma | Typ |
|---|---|---|
| 0️⃣ | Instalace | Příprava |
| 1️⃣ | Co je AI / ML? | Teorie |
| 2️⃣ | Tensory — základ PyTorch | Teorie + kód |
| 3️⃣ | Datasets & DataLoaders | Teorie + kód |
| 4️⃣ | Stavba neuronové sítě (`nn.Module`) | Teorie + kód |
| 5️⃣ | Trénovací smyčka (training loop) | Teorie + kód |
| 6️⃣ | Uložení a načtení modelu | Teorie + kód |
| 7️⃣ | Kompletní příklad: MNIST klasifikace | Praktický projekt |
| 8️⃣ | Vizualizace výsledků | Teorie + kód |
| 9️⃣ | CNN — konvoluční sítě | 🎯 Samostatná práce |
| 🔟 | Vlastní dataset | 🎯 Samostatná práce |
| ⭐ | Bonus: Transfer Learning | 🎯 Samostatná práce |
| ❌ | Časté chyby a ladění | Referenční sekce |

---

📖 **Oficiální dokumentace:** https://docs.pytorch.org/tutorials/beginner/basics/intro.html

---
# 0️⃣ Instalace

Nejprve nainstalujeme potřebné knihovny. PyTorch se instaluje z oficiálního kanálu.

> ⚠️ **Poznámka:** Pokud máte NVIDIA GPU a chcete akceleraci, nainstalujte verzi s CUDA — viz https://pytorch.org/get-started/locally/. Níže je CPU verze, která funguje všude.

## Co instalujeme a proč?

| Knihovna | K čemu slouží |
|---|---|
| **`torch`** | Hlavní framework PyTorch — obsahuje tensory, neuronové sítě, optimalizátory a vše potřebné pro strojové učení |
| **`torchvision`** | Rozšíření PyTorch pro práci s obrázky — hotové datasety (MNIST, CIFAR…), předtrénované modely a transformace obrázků |
| **`matplotlib`** | Knihovna pro vykreslování grafů a vizualizací — budeme ji používat pro zobrazení obrázků, grafů loss a accuracy |
| **`numpy`** | Základní knihovna pro práci s poli a maticemi — PyTorch na ní interně staví a umí s ní spolupracovat |
| **`pillow`** | Knihovna pro načítání a úpravu obrázků (formáty PNG, JPG…) — potřebujeme ji pro predikci z vlastních obrázků |

> 💡 **Co je to knihovna?** Knihovna (library) je sada předpřipravených funkcí a tříd, které si nemusíte psát sami. Místo psaní tisíců řádků kódu pro neuronovou síť stačí napsat `import torch` a máte k dispozici vše potřebné.

In [ ]:
# Instalace knihoven
!pip install torch torchvision matplotlib numpy pillow

In [ ]:
# Ověření instalace
import torch
print(f"PyTorch verze: {torch.__version__}")
print(f"CUDA dostupná: {torch.cuda.is_available()}")

# Výběr zařízení — GPU pokud je dostupná, jinak CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používané zařízení: {device}")

---
# 1️⃣ Co je AI / ML?

## Základní pojmy

| Pojem | Vysvětlení |
|---|---|
| **AI** (Artificial Intelligence) | Umělá inteligence — stroj napodobuje lidské myšlení |
| **ML** (Machine Learning) | Strojové učení — stroj se učí z dat, ne z pravidel |
| **DL** (Deep Learning) | Hluboké učení — ML pomocí neuronových sítí s mnoha vrstvami |
| **Neuronová síť** | Matematický model inspirovaný lidským mozkem |
| **Trénování** | Proces, kdy model opakovaně vidí data a zlepšuje své předpovědi |
| **Epoch** | Jeden průchod přes celý trénovací dataset |
| **Batch** | Malá skupina vzorků zpracovaná najednou |
| **Loss (ztráta)** | Číslo měřící, jak moc se model mýlí |
| **Optimalizátor** | Algoritmus, který upravuje váhy modelu, aby snižoval loss |

## Jak to funguje?

```
     ┌─────────┐      ┌──────────┐      ┌───────────┐
     │  DATA    │ ───► │  MODEL   │ ───► │ PREDIKCE  │
     │ (vstup)  │      │ (síť)    │      │ (výstup)  │
     └─────────┘      └──────────┘      └───────────┘
                            ▲
                            │
                      ┌──────────┐
                      │  LOSS    │ ◄── porovnání s pravdou
                      │ (chyba)  │
                      └──────────┘
                            │
                      ┌──────────────┐
                      │ OPTIMALIZÁTOR│ ── upraví váhy modelu
                      └──────────────┘
```

## Typy strojového učení

| Typ | Popis | Příklad |
|---|---|---|
| **Supervised** (s učitelem) | Máme data i správné odpovědi | Rozpoznávání číslic (MNIST) |
| **Unsupervised** (bez učitele) | Máme jen data, hledáme vzory | Shlukování zákazníků |
| **Reinforcement** (posilované) | Agent se učí z odměn/trestů | Hraní her (AlphaGo) |

> 💡 **V tomto kurzu** se budeme zabývat **supervised learning** — klasifikací obrázků.

---
# 2️⃣ Tensory — základ PyTorch

**Tensor** je základní datová struktura v PyTorch — podobná NumPy poli, ale s podporou GPU a automatického derivování.

## Proč potřebujeme tensory?

Počítač nerozumí obrázkům, textu ani zvuku přímo. Vše musíme převést na **čísla** — a právě tensory jsou způsob, jak tato čísla uspořádat.

> 🧠 **Představte si to takto:**
> - **Jedno číslo** (skalár) = teplota: `21.5°C`
> - **Řada čísel** (vektor) = teploty za týden: `[20, 21, 19, 22, 23, 21, 20]`
> - **Tabulka čísel** (matice) = černobílý obrázek — každý pixel je číslo 0–255
> - **Kvádr čísel** (3D tensor) = barevný obrázek — 3 tabulky (červená, zelená, modrá)
> - **Balík kvádrů** (4D tensor) = batch obrázků — více obrázků najednou

## Co je tensor?

| Dimenze | Název | Příklad | Analogie |
|---|---|---|---|
| 0D | Skalár | `tensor(42)` — jedno číslo | Jedna teplota |
| 1D | Vektor | `tensor([1, 2, 3])` — řada čísel | Řádek v tabulce |
| 2D | Matice | `tensor([[1,2],[3,4]])` — tabulka | Černobílý obrázek |
| 3D | 3D tensor | Obrázek (výška × šířka × kanály) | Barevný obrázek (RGB) |
| 4D | 4D tensor | Batch obrázků (batch × kanály × výška × šířka) | Balík obrázků pro trénink |

## Proč ne NumPy?

Tensor je velmi podobný NumPy poli, ale má **dvě klíčové výhody**:
1. **GPU akcelerace** — tensor lze přesunout na grafickou kartu, která počítá paralelně a **desítky× rychleji**
2. **Autograd** — PyTorch si automaticky pamatuje, jaké operace s tensorem proběhly, a umí spočítat **derivace** (gradienty) — to je základ trénování neuronových sítí

## Vytváření tensorů

```python
# Z dat
x = torch.tensor([1, 2, 3])         

# Z NumPy
x = torch.from_numpy(np_array)       

# Speciální
x = torch.zeros(3, 4)    # samé nuly — matice 3 řádky × 4 sloupce
x = torch.ones(3, 4)     # samé jedničky
x = torch.rand(3, 4)     # náhodné hodnoty mezi 0 a 1
```

## Vlastnosti tensoru

Každý tensor má tři důležité vlastnosti:

```python
tensor.shape    # rozměry — jak je tensor "velký" (např. [3, 4] = 3 řádky, 4 sloupce)
tensor.dtype    # datový typ — jakého typu jsou čísla (float32 = desetinná, int64 = celá)
tensor.device   # kde tensor žije — na CPU (procesoru) nebo CUDA (grafické kartě)
```

> 💡 **Shape (tvar)** je nejdůležitější vlastnost! Většina chyb v PyTorch vzniká kvůli nesouhlasícím tvarům tensorů.

### Kód — práce s tensory

In [ ]:
import torch
import numpy as np

# === Vytváření tensorů ===

# Z Python seznamu
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)
print(f"Z dat:\n{x_data}\n")

# Z NumPy pole
np_array = np.array([5, 6, 7])
x_np = torch.from_numpy(np_array)
print(f"Z NumPy: {x_np}\n")

# Speciální tensory
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
rand = torch.rand(2, 3)

print(f"Nuly:\n{zeros}\n")
print(f"Jedničky:\n{ones}\n")
print(f"Náhodné:\n{rand}\n")

In [ ]:
# === Vlastnosti tensorů ===
tensor = torch.rand(3, 4)

print(f"Tvar (shape): {tensor.shape}")
print(f"Datový typ:   {tensor.dtype}")
print(f"Zařízení:     {tensor.device}")

In [ ]:
# === Operace s tensory ===

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# Sčítání
print(f"a + b = {a + b}")

# Násobení po prvcích (element-wise)
print(f"a * b = {a * b}")

# Maticové násobení
mat_a = torch.rand(2, 3)
mat_b = torch.rand(3, 2)
print(f"\nMaticové násobení (2×3) @ (3×2):\n{mat_a @ mat_b}")

# Indexování (jako NumPy)
tensor = torch.ones(4, 4)
tensor[:, 1] = 0    # druhý sloupec = 0
print(f"\nIndexování:\n{tensor}")

# Spojování (cat)
t1 = torch.cat([tensor, tensor], dim=1)  # vedle sebe
print(f"\nSpojení (dim=1):\n{t1}")

# Přesun na GPU (pokud dostupná)
if torch.cuda.is_available():
    tensor_gpu = tensor.to("cuda")
    print(f"\nTensor na GPU: {tensor_gpu.device}")

### Co jsme právě dělali?

| Operace | Co dělá | Příklad |
|---|---|---|
| `a + b` | **Sčítání po prvcích** — sečte odpovídající prvky | `[1,2,3] + [4,5,6] = [5,7,9]` |
| `a * b` | **Násobení po prvcích** — vynásobí odpovídající prvky | `[1,2,3] * [4,5,6] = [4,10,18]` |
| `a @ b` | **Maticové násobení** — lineární algebra, základ neuronových sítí | Výsledek má jiný tvar! |
| `tensor[:, 1]` | **Indexování** — výběr konkrétních řádků/sloupců (jako v NumPy) | Vyber 2. sloupec |
| `torch.cat(...)` | **Spojování** — slepí tensory dohromady | Dva vedle sebe |
| `.to("cuda")` | **Přesun na GPU** — tensor se přesune do paměti grafické karty | Rychlejší výpočty |

> 💡 **Tip:** Tensor a NumPy pole mohou sdílet paměť — změna jednoho se projeví v druhém:
> ```python
> t = torch.ones(3)
> n = t.numpy()       # sdílí paměť!
> t.add_(1)           # in-place operace (podtržítko _ = in-place)
> print(n)            # [2. 2. 2.] — změnilo se i NumPy pole
> ```
>
> ⚠️ Funkce s podtržítkem na konci (např. `add_()`, `mul_()`) mění tensor **na místě** — nepotřebujete výsledek přiřazovat do nové proměnné.

---
# 3️⃣ Datasets & DataLoaders

PyTorch odděluje **data** od **modelu** — to je dobrá praxe. Používáme dva klíčové nástroje:

| Třída | Účel |
|---|---|
| `Dataset` | Ukládá vzorky a jejich popisky (labels) |
| `DataLoader` | Iterátor — dávkuje data, míchá je, paralelizuje načítání |

## Proč potřebujeme Dataset a DataLoader?

> 🧠 **Představte si školu:**
> - **Dataset** = učebnice se všemi příklady a správnými odpověďmi
> - **DataLoader** = učitel, který vám zadává příklady po skupinkách (batche), zamíchá je, aby se neopakovaly ve stejném pořadí

Bez DataLoaderu bychom museli ručně:
1. Vybírat vzorky z datasetu
2. Skládat je do skupin (batchů)
3. Míchat pořadí
4. Přesouvat data na GPU

DataLoader tohle **vše dělá automaticky**.

## Hotové datasety v `torchvision`

PyTorch nabízí řadu připravených datasetů (nemusíte shánět data sami):
- **MNIST** — ručně psané číslice 0–9 (28×28 px, 70 000 vzorků) — nejklasičtější dataset pro učení
- **FashionMNIST** — obrázky oblečení (28×28 px, 70 000 vzorků) — těžší než MNIST
- **CIFAR-10** — barevné fotky 10 kategorií (32×32 px) — ještě těžší

## Co je to „transform"?

Obrázky nelze přímo dát do neuronové sítě. Musíme je **transformovat** (předzpracovat):

| Krok | Co se děje | Proč |
|---|---|---|
| Načtení | Obrázek je ve formátu PIL (jako v prohlížeči fotek) | Nelze přímo počítat |
| `ToTensor()` | Převod na tensor + normalizace hodnot z 0–255 na 0.0–1.0 | Síť pracuje s tensory a malými čísly |
| Změna os | Z `(výška, šířka, kanály)` na `(kanály, výška, šířka)` | PyTorch očekává kanál jako první dimenzi |

## Co je batch a batch_size?

Nemůžeme dát do modelu všech 60 000 obrázků najednou — nevejdou se do paměti! Proto je dělíme na **batche** (dávky):

- `batch_size=64` → model vždy zpracuje 64 obrázků najednou
- Menší batch = méně paměti, ale pomalejší trénování
- Větší batch = rychlejší, ale potřebuje více paměti
- Typické hodnoty: **32, 64, 128, 256**

## Syntaxe

```python
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

# Stažení a příprava dat
train_data = datasets.MNIST(
    root="data",         # kam uložit soubory na disku
    train=True,          # trénovací část (60 000 vzorků)
    download=True,       # stáhnout z internetu pokud ještě nemáme
    transform=ToTensor() # převod obrázku na tensor
)

# DataLoader — dávkování a míchání
train_loader = DataLoader(
    train_data, 
    batch_size=64,   # po kolika vzorcích najednou
    shuffle=True     # zamíchat pořadí v každé epoše
)
```

## Proč rozdělujeme data na trénovací a testovací?

- **Trénovací data** (train) — z těchto se model učí
- **Testovací data** (test) — na těchto ověřujeme, jak dobře se model naučil

> ⚠️ Model **nikdy nevidí testovací data** během trénování! Jinak bychom nevěděli, jestli se opravdu naučil rozpoznávat vzory, nebo si jen „zapamatoval" konkrétní obrázky.

### Kód — načtení MNIST datasetu

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

# Stažení MNIST datasetu
train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

print(f"Trénovacích vzorků: {len(train_data)}")
print(f"Testovacích vzorků: {len(test_data)}")
print(f"Tvar jednoho obrázku: {train_data[0][0].shape}")
print(f"Label prvního vzorku: {train_data[0][1]}")

In [ ]:
# Zobrazení vzorků z datasetu
figure = plt.figure(figsize=(10, 4))
for i in range(10):
    # Najdeme vzorek s požadovanou číslicí
    idx = next(j for j, (_, label) in enumerate(train_data) if label == i)
    img, label = train_data[idx]
    
    figure.add_subplot(2, 5, i + 1)
    plt.title(f"Číslice: {label}")
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")

plt.tight_layout()
plt.show()

In [ ]:
# Vytvoření DataLoaderů
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Ukázka jednoho batche
images, labels = next(iter(train_loader))
print(f"Batch obrázků — shape: {images.shape}")   # [64, 1, 28, 28]
print(f"Batch labelů  — shape: {labels.shape}")    # [64]

> 💡 **Co dělá `ToTensor()`?**
> - Převede PIL obrázek (hodnoty pixelů 0–255, celá čísla) na PyTorch tensor (hodnoty 0.0–1.0, desetinná čísla)
> - Změní pořadí os: z `(H, W, C)` na `(C, H, W)` — kanál jako první dimenze
> - Proč normalizace na 0–1? Neuronové sítě pracují lépe s **malými čísly** — velké hodnoty (0–255) mohou způsobit nestabilitu při trénování
>
> 💡 **Co je `shuffle=True`?**
> - Při **každé epoše** se pořadí vzorků náhodně zamíchá
> - Zabraňuje tomu, aby se model „naučil pořadí" místo vzorů v datech
> - Příklad: pokud jsou data seřazená (všechny 0, pak 1, pak 2…), model by se mohl naučit: „prvních 6000 vzorků = 0", což je špatně!
>
> 💡 **Proč `shuffle=False` u testovacích dat?**
> - Při testování nám na pořadí nezáleží — jen měříme přesnost
> - Výsledky jsou **reprodukovatelné** (stejné při každém spuštění)

---
# 4️⃣ Stavba neuronové sítě (`nn.Module`)

V PyTorch definujeme neuronovou síť jako **třídu** dědící z `nn.Module`.

## Co je neuronová síť?

Neuronová síť je matematická funkce, která se skládá z **vrstev**. Každá vrstva:
1. Vezme vstupní čísla (tensor)
2. Provede s nimi výpočty (násobení vahami + přičtení biasu)
3. Výsledek pošle do další vrstvy

> 🧠 **Analogie:** Představte si továrnu s několika dílnami (vrstvami). Surovina (obrázek) projde první dílnou, ta ji trochu zpracuje a pošle dál. Každá dílna přidá něco svého. Na konci z továrny vyjde hotový výrobek (predikce).

## Co jsou váhy (weights) a bias?

Každá vrstva má **parametry**, které se model učí upravovat:

- **Váhy (weights)** — čísla, kterými se násobí vstupy. Určují, jak moc je který vstup důležitý.
- **Bias** — číslo, které se přičte k výsledku. Umožňuje posunout výstup.

> 🧠 **Analogie:** Váhy jsou jako „důležitost" — pokud chcete rozpoznat číslici „1", horní pixel je méně důležitý než svislá čára uprostřed. Bias je jako „posun nuly" — některé neurony potřebují být aktivní i bez vstupů.

Na začátku jsou váhy **náhodné** — model nic neumí. Během trénování se váhy postupně upravují tak, aby model dělal co nejméně chyb.

## Základní vrstvy

| Vrstva | Popis | Co dělá konkrétně |
|---|---|---|
| `nn.Flatten()` | Převede 2D obrázek na 1D vektor | 28×28 obrázek → 784 čísel v řadě |
| `nn.Linear(in, out)` | Plně propojená vrstva | Každý vstup je spojený s každým výstupem (násobení maticí vah) |
| `nn.ReLU()` | Aktivační funkce | Záporné hodnoty změní na 0, kladné nechá |
| `nn.Softmax(dim=1)` | Převod na pravděpodobnosti | Výstupy se přepočítají tak, aby se sečetly na 1.0 |
| `nn.Sequential(...)` | Kontejner | Vrstvy se provedou postupně za sebou |

## Proč potřebujeme aktivační funkce (ReLU)?

Bez aktivační funkce by neuronová síť byla jen **jedno velké násobení maticí** — to je jen lineární transformace a zvládne jen jednoduché problémy (přímky).

Aktivační funkce přidává **nelinearitu** — díky tomu se síť naučí i složité vzory (křivky, hrany, textury…).

**ReLU (Rectified Linear Unit)** je nejpoužívanější aktivace:
```
ReLU(x) = max(0, x)

Vstup:  [-2, -1, 0, 1, 2, 3]
Výstup: [ 0,  0, 0, 1, 2, 3]   ← záporné → 0, kladné beze změny
```

> 💡 Proč zrovna ReLU? Je **jednoduchá**, **rychlá** a **funguje skvěle** v praxi. Existují i jiné (Sigmoid, Tanh, LeakyReLU…), ale ReLU je nejčastější volba.

## Co dělá metoda `forward()`?

Metoda `forward()` definuje, jak data **protečou** sítí — tedy jaké vrstvy se na ně aplikují a v jakém pořadí. Toto se nazývá **forward pass** (dopředný průchod).

## Struktura třídy

```python
class MujModel(nn.Module):
    def __init__(self):
        super().__init__()          # inicializace rodičovské třídy
        # Definice vrstev (jaké "dílny" naše továrna má)
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(784, 128),    # 784 vstupů → 128 výstupů
            nn.ReLU(),              # aktivace
            nn.Linear(128, 10),     # 128 vstupů → 10 výstupů (10 číslic)
        )
    
    def forward(self, x):
        # Jak data prochází sítí (pořadí zpracování)
        x = self.flatten(x)         # obrázek → vektor
        logits = self.layers(x)     # vektor → predikce
        return logits
```

> ⚠️ **Důležité:** Nikdy nevolejte `model.forward(x)` přímo! Vždy `model(x)` — PyTorch interně přidá další operace (např. hooks pro sledování).

## Jak vypadá naše síť vizuálně?

```
Vstup: obrázek 28×28 pixelů
        │
   ┌────┴────┐
   │ Flatten  │  → 784 hodnot (28×28 = 784)
   └────┬────┘
        │
   ┌────┴────────┐
   │ Linear(784→128) │  → 784 vstupů se zredukuje na 128
   └────┬────────┘
        │
   ┌────┴────┐
   │  ReLU   │  → záporné hodnoty → 0
   └────┬────┘
        │
   ┌────┴────────┐
   │ Linear(128→64)  │  → dalí zmenšení
   └────┬────────┘
        │
   ┌────┴────┐
   │  ReLU   │
   └────┬────┘
        │
   ┌────┴────────┐
   │ Linear(64→10)   │  → 10 výstupů = 10 číslic (0–9)
   └────┬────────┘
        │
   Výstup: 10 čísel (logits) — skóre pro každou číslici
```

### Kód — definice modelu

In [ ]:
import torch
from torch import nn

class MNISTModel(nn.Module):
    """Jednoduchá neuronová síť pro klasifikaci číslic MNIST."""
    
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()         # 28×28 → 784
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),        # 784 → 128
            nn.ReLU(),                       # aktivace
            nn.Linear(128, 64),              # 128 → 64
            nn.ReLU(),                       # aktivace
            nn.Linear(64, 10),               # 64 → 10 (10 číslic)
        )
    
    def forward(self, x):
        x = self.flatten(x)                  # zploštění obrázku
        logits = self.layers(x)              # průchod vrstvami
        return logits                        # vrátíme surové skóre (logits)

# Vytvoření modelu a přesun na zařízení
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MNISTModel().to(device)
print(model)
print(f"\nPočet parametrů: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test — pošleme jeden náhodný "obrázek" sítí
x_test = torch.rand(1, 1, 28, 28, device=device)  # 1 obrázek, 1 kanál, 28×28
logits = model(x_test)
pred_probs = nn.Softmax(dim=1)(logits)

print(f"Logits (surové skóre):    {logits.detach().cpu()}")
print(f"Pravděpodobnosti:         {pred_probs.detach().cpu()}")
print(f"Predikovaná třída:        {pred_probs.argmax(1).item()}")

> 💡 **Logits vs. pravděpodobnosti — co to je?**
>
> - **Logits** = surové výstupy sítě — „skóre" pro každou třídu. Mohou být záporné i velké. Nemají žádný jasný rozsah.
>   - Příklad: `[2.1, -0.5, 0.3, ..., 8.7]` → čím vyšší číslo, tím si model myslí, že to je ta číslice
> - **Softmax** = matematická funkce, která převede logits na **pravděpodobnosti** (čísla mezi 0 a 1, součet = 1)
>   - Příklad: `[0.01, 0.00, 0.02, ..., 0.95]` → model si myslí na 95%, že to je daná číslice
> - **`argmax`** = najde pozici (index) s **nejvyšší hodnotou** — to je naše predikce
>   - Příklad: `argmax([0.01, 0.00, 0.02, ..., 0.95])` → index `9` → model říká „to je devítka"
>
> ⚠️ Při **trénování** používáme logits + `CrossEntropyLoss` (ta aplikuje softmax interně — neaplikujte ho dvakrát!)
>
> ⚠️ Při **predikci** použijeme Softmax ručně, abychom viděli pravděpodobnosti.

---
# 5️⃣ Trénovací smyčka (Training Loop)

Trénování modelu se skládá ze dvou částí, které se opakují v každé epoše:

## Co je epocha?

**Epocha** = jeden kompletní průchod přes **všechna** trénovací data. Pokud máme 60 000 obrázků a `batch_size=64`, jedna epocha má přibližně 938 batchů (60000 ÷ 64 ≈ 938).

> 🧠 **Analogie:** Epocha je jako když si přečtete **celou učebnici** jednou. Po 10 epochách jste ji přečetli 10×. Pokaždé se naučíte něco víc, ale po příliš mnoha opakováních se začnete učit nazpaměť (overfitting).

## Trénovací cyklus

```
Pro každou epochu:
  1. TRÉNOVÁNÍ (train loop):
     - Vezmi batch dat (64 obrázků)
     - Pošli je modelem → predikce
     - Spočítej loss (jak moc se model mýlí)
     - Zpětná propagace (backpropagation) → spočítej gradienty
     - Aktualizuj váhy (optimizer.step) → model se zlepší
  
  2. TESTOVÁNÍ (test loop):
     - Vyhodnoť model na testovacích datech (která nikdy neviděl)
     - Spočítej accuracy (přesnost) — kolik obrázků uhádl správně
```

## Klíčové komponenty

### Loss funkce (ztrátová funkce)

Loss (ztráta) je **číslo, které měří, jak moc se model mýlí**. Čím menší loss, tím lépe.

| Loss funkce | Kdy ji použít | Příklad |
|---|---|---|
| `nn.CrossEntropyLoss()` | Klasifikace (výběr z N tříd) | Rozpoznávání číslic 0–9 |
| `nn.MSELoss()` | Regrese (predikce čísla) | Předpověď ceny bytu |

> 🧠 **CrossEntropyLoss** jednoduše řečeno: pokud model řekne „jsem si na 90% jistý, že to je 5" a skutečně to je 5, loss je **malý**. Pokud si model myslí, že to je 3, ale správně je 5, loss je **velký**.

### Optimalizátor

Optimalizátor je algoritmus, který **upravuje váhy modelu** na základě gradientů tak, aby se loss snižoval.

| Optimalizátor | Popis |
|---|---|
| `torch.optim.SGD(...)` | Stochastic Gradient Descent — jednoduchý, spolehlivý |
| `torch.optim.Adam(...)` | Adaptivní — rychlejší konvergence, dobrá výchozí volba |

### Learning rate (rychlost učení)

Learning rate určuje, **jak velké kroky** optimalizátor dělá při úpravě vah.

- **Příliš velký** (např. 0.1) → model „přeskakuje" optimum, loss skáče nahoru a dolů
- **Příliš malý** (např. 0.000001) → model se učí extrémně pomalu
- **Správný** (typicky 0.001) → model plynule konverguje k dobrému řešení

> 🧠 **Analogie:** Představte si, že hledáte nejnižší místo v údolí se zavázanýma očima. Learning rate je **velikost vašeho kroku**. Příliš velký krok → přeskočíte údolí. Příliš malý → trvá to věčnost.

## Co je zpětná propagace (backpropagation)?

Zpětná propagace je matematický postup, jak zjistit, **které váhy jsou zodpovědné za chybu** a **jak je upravit**.

1. **Forward pass** — data projdou sítí dopředu → dostaneme predikci
2. **Loss** — spočítáme chybu (jak moc se model mýlí)
3. **Backward pass** — od chyby se jdeme **zpět** sítí a počítáme **gradienty** (derivace)
4. **Gradient** říká: „tato váha by se měla zvýšit/snížit o tolik, aby se loss zmenšil"
5. **Optimizer.step()** — skutečně váhy upraví podle gradientů

## 3 kroky v každém batchi:

```python
optimizer.zero_grad()    # 1. Vynuluj gradienty z předchozího batche
loss.backward()          # 2. Zpětná propagace — spočítej, jak upravit váhy
optimizer.step()         # 3. Uprav váhy podle spočítaných gradientů
```

> ⚠️ **Proč `zero_grad()`?** PyTorch **přičítá** gradienty k předchozím. Pokud je nevynulujeme, kumulují se z předchozích batchů a trénování nefunguje správně!

### Kód — trénovací a testovací smyčka

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, device):
    """Jedna epocha trénování."""
    model.train()  # přepnout do trénovacího režimu
    total_loss = 0
    
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)  # přesun na GPU/CPU
        
        pred = model(X)           # 1. predikce
        loss = loss_fn(pred, y)   # 2. výpočet chyby
        
        loss.backward()           # 3. zpětná propagace
        optimizer.step()          # 4. aktualizace vah
        optimizer.zero_grad()     # 5. vynulování gradientů
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    return avg_loss


def test_loop(dataloader, model, loss_fn, device):
    """Vyhodnocení modelu na testovacích datech."""
    model.eval()  # přepnout do evaluačního režimu
    test_loss, correct = 0, 0
    
    with torch.no_grad():  # nepočítáme gradienty
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
    
    test_loss /= len(dataloader)
    accuracy = correct / len(dataloader.dataset)
    return test_loss, accuracy

> 💡 **`model.train()` vs `model.eval()` — proč to musím přepínat?**
>
> Některé vrstvy (Dropout, BatchNorm) se chovají **jinak** při trénování a při testování:
>
> | Režim | Co se děje |
> |---|---|
> | `model.train()` | Dropout **náhodně vypíná** neurony (aby model nebyl přetrénovaný). BatchNorm počítá statistiky z aktuálního batche. |
> | `model.eval()` | Dropout je **vypnutý** (používají se všechny neurony). BatchNorm používá celkové statistiky naučené při tréninku. |
>
> - `torch.no_grad()` — říká PyTorchu: „nepočítej a neukládej gradienty". Šetří paměť a zrychluje výpočet. Používáme v testovací smyčce, protože tam váhy neupravujeme.
>
> ⚠️ **Zapomenutí `model.eval()`** při testování je **častá chyba** — výsledky budou horší a nestabilní!

---
# 6️⃣ Uložení a načtení modelu

Po natrénování chceme model uložit, abychom ho mohli použít později bez opětovného trénování.

> 🧠 **Proč ukládat model?** Trénování může trvat minuty, hodiny i dny. Nechcete ho opakovat pokaždé, když chcete provést predikci. Uložený model můžete sdílet s ostatními nebo nasadit do aplikace.

## Co se vlastně ukládá?

Model se skládá z:
1. **Architektury** — jaké vrstvy a jak jsou propojené (to máme v kódu — definice třídy)
2. **Vah (state_dict)** — naučené parametry (tisíce čísel, které se model naučil během trénování)

## Dva způsoby ukládání

| Způsob | Příkaz | Co se uloží | Výhoda |
|---|---|---|---|
| **Jen váhy** ✅ | `torch.save(model.state_dict(), 'model.pth')` | Jen naučené váhy | Menší soubor, flexibilnější |
| **Celý model** | `torch.save(model, 'model.pth')` | Vrstvy + váhy | Jednodušší načtení, ale závisí na pickle |

> 💡 Doporučuje se ukládat **jen váhy** (`state_dict`) — je to bezpečnější a flexibilnější.

## Načtení vah

```python
# 1. Vytvořit prázdný model (stejná architektura jako při trénování)
model = MNISTModel()

# 2. Načíst uložené váhy do modelu
model.load_state_dict(torch.load('model.pth', weights_only=True))

# 3. Přepnout do evaluačního režimu (vypne Dropout atd.)
model.eval()
```

> ⚠️ **Vždy** volejte `model.eval()` před inferencí (predikcí)!
>
> ⚠️ Architektura modelu (třída) musí být **identická** při ukládání i načítání — jinak se váhy nenačtou!

---
# 7️⃣ Kompletní příklad: MNIST klasifikace

Nyní dáme vše dohromady — od načtení dat po predikci z vlastního obrázku.

> 🧠 **Co budeme dělat?** V předchozích sekcích jsme si vysvětlili jednotlivé součástky (tensory, data, model, trénování). Teď je **spojíme do jednoho celku** a natrénujeme skutečný model na rozpoznávání ručně psaných číslic.

## Postup
1. **Načteme** MNIST dataset (60 000 trénovacích + 10 000 testovacích obrázků)
2. **Vytvoříme** model, loss funkci a optimalizátor
3. **Natrénujeme** model (10 epoch — 10× projdeme celý dataset)
4. **Vyhodnotíme** přesnost na testovacích datech
5. **Uložíme** natrénovaný model na disk
6. **Otestujeme** predikci na vlastním obrázku

### Kód — kompletní trénování

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# ============================
# 1. PŘÍPRAVA DAT
# ============================
train_data = datasets.MNIST(root="data", train=True, download=True, transform=ToTensor())
test_data = datasets.MNIST(root="data", train=False, download=True, transform=ToTensor())

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# ============================
# 2. DEFINICE MODELU
# ============================
class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MNISTModel().to(device)

# ============================
# 3. LOSS A OPTIMALIZÁTOR
# ============================
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Model: {model}")
print(f"Zařízení: {device}")
print(f"Parametrů: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================
# 4. TRÉNOVÁNÍ
# ============================
epochs = 10
history = {"train_loss": [], "test_loss": [], "accuracy": []}

for epoch in range(epochs):
    # Trénování
    train_loss = train_loop(train_loader, model, loss_fn, optimizer, device)
    
    # Testování
    test_loss, accuracy = test_loop(test_loader, model, loss_fn, device)
    
    # Uložení do historie
    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["accuracy"].append(accuracy)
    
    print(f"Epocha {epoch+1:2d}/{epochs} | "
          f"Train loss: {train_loss:.4f} | "
          f"Test loss: {test_loss:.4f} | "
          f"Accuracy: {accuracy*100:.1f}%")

print(f"\n✅ Trénování dokončeno! Nejlepší přesnost: {max(history['accuracy'])*100:.1f}%")

In [ ]:
# ============================
# 5. ULOŽENÍ MODELU
# ============================
torch.save(model.state_dict(), "mnist_pytorch.pth")
print("Model uložen do mnist_pytorch.pth")

> 💡 **Adam vs SGD — jaký optimalizátor zvolit?**
>
> | Optimalizátor | Jak funguje | Kdy použít |
> |---|---|---|
> | **SGD** | Dělá stejně velké kroky pro všechny váhy | Jednoduchý, spolehlivý, ale pomalejší konvergence |
> | **Adam** | Přizpůsobuje velikost kroků pro **každou váhu zvlášť** | Rychlejší konvergence, dobrá výchozí volba pro začátečníky |
>
> Pro začátek doporučujeme **Adam** s `lr=0.001` — funguje spolehlivě ve většině případů.
>
> 💡 **Co je `lr` (learning rate)?** Viz sekce 5 — je to velikost kroků, kterými optimalizátor upravuje váhy. Hodnota `0.001` je osvědčený výchozí bod.

---
# 8️⃣ Vizualizace výsledků

Vizualizace je klíčová pro pochopení toho, jak se model učí a kde chybuje.

> 🧠 **Proč vizualizovat?**
> - **Graf loss** ukazuje, zda se model zlepšuje (loss klesá) nebo se něco pokazilo (loss roste/stagnuje)
> - **Graf accuracy** ukazuje, kolik procent obrázků model správně klasifikoval
> - **Ukázky predikcí** odhalí, na jakých obrázcích model chybuje (např. záměna 4 a 9)

### Kód — grafy trénování

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Graf loss
ax1.plot(history["train_loss"], label="Train loss", marker="o")
ax1.plot(history["test_loss"], label="Test loss", marker="s")
ax1.set_xlabel("Epocha")
ax1.set_ylabel("Loss")
ax1.set_title("Průběh loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Graf accuracy
ax2.plot([a * 100 for a in history["accuracy"]], label="Test accuracy", marker="o", color="green")
ax2.set_xlabel("Epocha")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Přesnost na testovacích datech")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Kód — predikce a vizualizace chyb

In [ ]:
# Zobrazení predikcí na náhodných testovacích vzorcích
model.eval()
figure = plt.figure(figsize=(12, 6))

for i in range(15):
    idx = torch.randint(len(test_data), size=(1,)).item()
    img, true_label = test_data[idx]
    
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))
        pred_label = pred.argmax(1).item()
    
    figure.add_subplot(3, 5, i + 1)
    color = "green" if pred_label == true_label else "red"
    plt.title(f"P:{pred_label} (S:{true_label})", color=color)
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")

plt.suptitle("Predikce (P) vs Skutečnost (S) — zelená=správně, červená=chyba", y=1.02)
plt.tight_layout()
plt.show()

### Kód — predikce z vlastního obrázku

Nakreslete číslici v kreslícím programu (bílá na černém pozadí, 28×28 px) a uložte jako PNG.

In [ ]:
from PIL import Image
import torchvision.transforms as transforms

def predict_image(model, image_path, device):
    """Načte obrázek a provede predikci."""
    # Načtení a předzpracování obrázku
    img = Image.open(image_path).convert("L")  # převod na stupně šedi
    
    transform = transforms.Compose([
        transforms.Resize((28, 28)),    # změna velikosti na 28×28
        transforms.ToTensor(),           # převod na tensor (0–1)
    ])
    
    img_tensor = transform(img).unsqueeze(0).to(device)  # přidáme batch dimenzi
    
    # Predikce
    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)
        probs = nn.Softmax(dim=1)(logits)
        pred = probs.argmax(1).item()
        confidence = probs.max().item()
    
    # Zobrazení
    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"Predikce: {pred} ({confidence*100:.1f}%)")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.bar(range(10), probs.cpu().squeeze().numpy())
    plt.xticks(range(10))
    plt.xlabel("Číslice")
    plt.ylabel("Pravděpodobnost")
    plt.title("Rozložení pravděpodobností")
    
    plt.tight_layout()
    plt.show()
    return pred

# Použití:
# predict_image(model, "moje_cislice.png", device)

> 💡 **Tip pro vlastní obrázky:**
> - MNIST má **bílé číslice na černém pozadí**
> - Pokud máte opačné barvy, invertujte: `img = 1.0 - img`
> - Ideální velikost je 28×28 pixelů
> - Použijte `Malování` nebo `Paint` pro jednoduché kreslení

---
# 9️⃣ 🎯 Samostatná práce: CNN — konvoluční neuronová síť

Výše jsme použili plně propojenou síť (Fully Connected / Dense). Teď vás čeká **konvoluční síť (CNN)** — ta je mnohem lepší pro práci s obrázky!

## Proč je plně propojená síť špatná pro obrázky?

V plně propojené síti (Linear) je **každý pixel spojený s každým neuronem**. To má dva problémy:

1. **Ignoruje prostorové vztahy** — síť neví, že pixel nahoře vlevo a pixel nahoře vpravo spolu sousedí. Každý pixel je pro ni jen číslo v dlouhém seznamu.
2. **Příliš mnoho parametrů** — pro obrázek 28×28 = 784 vstupů × 128 neuronů = přes 100 000 vah jen v první vrstvě! U větších obrázků to roste drasticky.

> 🧠 **Analogie:** Představte si, že hledáte kočku na fotce. Plně propojená síť se dívá na **každý pixel zvlášť**. CNN se dívá na **malé oblasti** (3×3 pixely) a hledá vzory — hrany, textury, tvary — **bez ohledu na to, kde na obrázku jsou**.

## Teorie — jak CNN funguje?

### Konvoluce (Conv2d) — hledání vzorů

Konvoluční vrstva používá malý **filtr** (kernel), který se posouvá po celém obrázku a hledá vzory:

```
Filtr 3×3 (detekce svislé hrany):      Obrázek:
┌────────────┐                          ┌─────────────────┐
│ -1   0   1 │                          │ 0 0 1 1 1 0 0 0 │
│ -1   0   1 │  ──posouvá se po──►     │ 0 0 1 1 1 0 0 0 │
│ -1   0   1 │                          │ 0 0 1 1 1 0 0 0 │
└────────────┘                          │ 0 0 1 1 1 0 0 0 │
                                        └─────────────────┘
```

- Filtr se **přiloží** na oblast 3×3 v obrázku
- Hodnoty se **vynásobí a sečtou** → jedno číslo
- Filtr se posune o 1 pixel → další číslo
- Výsledek je nový obrázek (**feature mapa**) ukazující, kde se vzor nachází

> 💡 Model se **sám naučí**, jaké filtry potřebuje! Na začátku jsou náhodné, ale po trénování detekují hrany, rohy, textury atd.

### Pooling (MaxPool2d) — zmenšení

MaxPool2d **zmenší** obrázek tím, že z každé oblasti 2×2 vezme **maximum**:

```
Vstup 4×4:              Po MaxPool2d(2):
┌───┬───┬───┬───┐       ┌───┬───┐
│ 1 │ 3 │ 2 │ 1 │       │ 4 │ 6 │   (max z levého horního 2×2 = 4)
│ 4 │ 2 │ 6 │ 4 │  ──►  │ 8 │ 5 │   (max z pravého horního 2×2 = 6)
│ 7 │ 8 │ 3 │ 5 │       └───┴───┘
│ 1 │ 0 │ 2 │ 1 │
└───┴───┴───┴───┘
```

Proč pooling? **Zmenší data** (méně výpočtů) a udělá síť **odolnější** vůči malým posunům v obrázku.

## Vrstvy CNN

| Vrstva | Co dělá | PyTorch |
|---|---|---|
| **Conv2d** | Detekuje vzory (hrany, textury…) pomocí filtrů | `nn.Conv2d(in_channels, out_channels, kernel_size)` |
| **ReLU** | Aktivace — záporné → 0 | `nn.ReLU()` |
| **MaxPool2d** | Zmenší obrázek — vezme maximum z oblasti | `nn.MaxPool2d(kernel_size)` |
| **Flatten** | Převede 2D feature mapy na 1D vektor | `nn.Flatten()` |
| **Linear** | Plně propojená vrstva pro finální klasifikaci | `nn.Linear(in, out)` |

## Jak data prochází CNN?

```
Obrázek 28×28 (1 kanál — černobílý)
    │
    ▼
Conv2d(1, 32, 3)  → 32 feature map, 26×26   ← 32 různých filtrů hledá 32 různých vzorů
    │                                           (26 = 28-3+1, filtr 3×3 zmenší o 2)
ReLU                                          ← záporné hodnoty → 0
    │
MaxPool2d(2)      → 32 feature map, 13×13    ← zmenšení na polovinu (26÷2 = 13)
    │
Conv2d(32, 64, 3) → 64 feature map, 11×11    ← 64 filtrů hledá složitější vzory
    │
ReLU
    │
MaxPool2d(2)      → 64 feature map, 5×5      ← další zmenšení (11÷2 = 5, zaokr. dolů)
    │
Flatten            → 1600 hodnot              ← 64 × 5 × 5 = 1600
    │
Linear(1600, 128)                             ← klasická plně propojená vrstva
    │
Linear(128, 10)   → 10 tříd (číslice 0–9)    ← finální predikce
```

> 💡 **Shrnutí:** Konvoluční vrstvy **extrahují vzory** z obrázku (hrany → tvary → části objektů). Plně propojené vrstvy na konci **klasifikují** — rozhodnou, co na obrázku je.

## Zadání

1. **Vytvořte třídu `CNNModel(nn.Module)`** s architekturou výše
2. **Natrénujte** ji na MNIST datasetu (10 epoch)
3. **Porovnejte přesnost** s předchozím plně propojeným modelem
4. **Zobrazte grafy** loss a accuracy

### Nápověda — kostra CNN třídy

<details>
<summary>🔽 Klikni pro zobrazení kostry</summary>

```python
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Konvoluční část — extrakce vzorů z obrázku
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),   # (1, 28, 28) → (32, 26, 26)
            nn.ReLU(),
            nn.MaxPool2d(2),                    # (32, 26, 26) → (32, 13, 13)
            nn.Conv2d(32, 64, kernel_size=3),   # (32, 13, 13) → (64, 11, 11)
            nn.ReLU(),
            nn.MaxPool2d(2),                    # (64, 11, 11) → (64, 5, 5)
        )
        # Klasifikační část — rozhodnutí, co na obrázku je
        self.fc_layers = nn.Sequential(
            nn.Flatten(),                       # (64, 5, 5) → 1600
            nn.Linear(64 * 5 * 5, 128),         # 1600 → 128
            nn.ReLU(),
            nn.Linear(128, 10),                 # 128 → 10 (číslic)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)    # extrakce vzorů
        x = self.fc_layers(x)      # klasifikace
        return x
```
</details>

### Hodnotící kritéria

| Kritérium | Body |
|---|---|
| Funkční CNN model s Conv2d + MaxPool2d | 3b |
| Trénování na MNIST (≥95% accuracy) | 2b |
| Porovnání s plně propojeným modelem | 2b |
| Grafy loss a accuracy | 1b |
| Vlastní experimentování (jiné filtry, vrstvy…) | 2b |
| **Celkem** | **10b** |

In [ ]:
# === ZDE NAPIŠTE SVŮJ KÓD ===
# Tip: zkopírujte trénovací smyčku ze sekce 7 a vyměňte model za CNNModel


---
# 🔟 🎯 Samostatná práce: FashionMNIST — vlastní dataset

Naučili jste se rozpoznávat číslice. Teď zkuste **oblečení**!

## FashionMNIST

Stejný formát jako MNIST (28×28, šedotónové), ale místo číslic obsahuje 10 kategorií oblečení:

| Label | Kategorie | Label | Kategorie |
|---|---|---|---|
| 0 | T-Shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

> 🧠 **Proč FashionMNIST?** MNIST (číslice) je příliš jednoduchý — i základní modely dosahují 97%+. FashionMNIST je **těžší** a realističtější — model musí rozlišovat mezi podobnými kategoriemi (T-Shirt vs Shirt, Pullover vs Coat).

## Co je matice záměn (Confusion Matrix)?

Matice záměn ukazuje, **které třídy si model plete**:
- Řádky = skutečná třída
- Sloupce = predikovaná třída
- Číslo na pozici [i, j] = kolikrát model řekl „j", ale správně bylo „i"

> 🧠 **Příklad:** Pokud model často zaměňuje „Shirt" za „T-shirt", v matici záměn bude na odpovídající pozici **vysoké číslo**. To vám řekne, kde model potřebuje zlepšit.

## Zadání

1. **Načtěte FashionMNIST** dataset (místo `datasets.MNIST` použijte `datasets.FashionMNIST`)
2. **Zobrazte** ukázkové obrázky s popisky
3. **Natrénujte** model (můžete použít CNN ze sekce 9)
4. **Vyhodnoťte** přesnost a zobrazte **matici záměn** (confusion matrix)

### Nápověda — matice záměn

<details>
<summary>🔽 Klikni pro zobrazení kódu matice záměn</summary>

```python
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Sesbírat všechny predikce
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        pred = model(X).argmax(1).cpu()
        all_preds.extend(pred.numpy())
        all_labels.extend(y.numpy())

# Zobrazení
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=kategorie)
disp.plot(figsize=(10, 10), cmap="Blues")
plt.title("Matice záměn")
plt.show()
```
</details>

### Hodnotící kritéria

| Kritérium | Body |
|---|---|
| Načtení FashionMNIST + zobrazení vzorků | 2b |
| Funkční model a trénování (≥85% accuracy) | 3b |
| Vizualizace výsledků (grafy + predikce) | 2b |
| Matice záměn (confusion matrix) | 2b |
| Bonus: experiment s hyperparametry | 1b |
| **Celkem** | **10b** |

In [ ]:
# === ZDE NAPIŠTE SVŮJ KÓD ===


---
# ⭐ Bonus: Transfer Learning

**Transfer Learning** = vezmeme model předtrénovaný na velkém datasetu (např. ImageNet s miliony obrázků) a přizpůsobíme ho na náš menší problém.

> 🧠 **Analogie:** Představte si, že chcete naučit někoho rozpoznávat houby. Je jednodušší naučit to člověka, který **už umí rozpoznávat rostliny** (má znalost tvarů, barev, textur), než někoho, kdo nikdy nic podobného neviděl. Transfer learning je přesně tohle — vezmeme „znalosti" z jiného úkolu.

## Proč Transfer Learning?

| Výhoda | Popis |
|---|---|
| ⏱️ **Rychlejší trénování** | Většina vah je už naučená — neučíme od nuly |
| 📊 **Méně dat** | Funguje i s malým datasetem (stovky obrázků místo tisíců) |
| 🎯 **Lepší výsledky** | Využívá znalosti z milionů obrázků — rozumí hranám, texturám, tvarům |

## Co je ResNet?

**ResNet (Residual Network)** je populární architektura CNN vytvořená společností Microsoft. ResNet18 má 18 vrstev a byl natrénovaný na **ImageNet** (1,2 milionu obrázků, 1000 kategorií).

Spodní vrstvy (blíže vstupu) se naučily rozpoznávat **univerzální vzory** — hrany, rohy, textury. Tyto znalosti jsou užitečné pro **jakýkoli** úkol s obrázky.

## Postup

```
1. Načíst předtrénovaný model (např. ResNet18 s vahami z ImageNet)
2. "Zmrazit" všechny vrstvy (freeze) — nechceme měnit naučené vzory
3. Vyměnit poslední vrstvu za vlastní (nový počet tříd — místo 1000 jen 10)
4. Trénovat jen poslední vrstvu — ta se naučí klasifikovat do našich tříd
```

> 💡 **Co znamená "zmrazit" vrstvy?** Nastavíme `requires_grad = False` — PyTorch nebude počítat gradienty pro tyto vrstvy, takže se jejich váhy **nezmění** během trénování. Trénuje se jen nová poslední vrstva.

## Zadání (pro pokročilé)

1. Načtěte `torchvision.models.resnet18(weights='IMAGENET1K_V1')`
2. Zmrazte všechny parametry: `param.requires_grad = False`
3. Vyměňte `model.fc` za `nn.Linear(512, 10)`
4. Natrénujte na FashionMNIST (pozor — ResNet očekává 3 kanály a 224×224!)

<details>
<summary>🔽 Klikni pro nápovědu — úprava dat pro ResNet</summary>

```python
# ResNet byl natrénovaný na barevných obrázcích 224×224 s normalizací ImageNet
# Naše data (FashionMNIST) jsou šedotónová 28×28 — musíme je přizpůsobit:
transform = transforms.Compose([
    transforms.Resize((224, 224)),                          # zvětšit z 28×28 na 224×224
    transforms.Grayscale(num_output_channels=3),            # 1 kanál → 3 kanály (RGB)
    transforms.ToTensor(),                                   # převod na tensor
    transforms.Normalize([0.485, 0.456, 0.406],             # normalizace jako ImageNet
                         [0.229, 0.224, 0.225]),
])
```
</details>

<details>
<summary>🔽 Klikni pro nápovědu — kód transfer learningu</summary>

```python
import torchvision.models as models

# 1. Načíst předtrénovaný model (stáhne váhy z internetu)
model = models.resnet18(weights='IMAGENET1K_V1')

# 2. Zmrazit všechny vrstvy (nebudou se měnit při trénování)
for param in model.parameters():
    param.requires_grad = False

# 3. Vyměnit poslední vrstvu (fc = fully connected)
#    ResNet18 má na konci Linear(512, 1000) — my chceme jen 10 tříd
model.fc = nn.Linear(512, 10)

# 4. Trénovat jen novou vrstvu (model.fc.parameters())
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
```
</details>

In [ ]:
# === ZDE NAPIŠTE SVŮJ KÓD ===


---
# ❌ Časté chyby a ladění

## 1. `RuntimeError: mat1 and mat2 shapes cannot be multiplied`

**Příčina:** Rozměry tensoru nesedí mezi vrstvami.

**Řešení:** Zkontrolujte výstup předchozí vrstvy. Vložte `print(x.shape)` do metody `forward()`:
```python
def forward(self, x):
    print(f"Vstup: {x.shape}")
    x = self.flatten(x)
    print(f"Po flatten: {x.shape}")
    ...
```

---

## 2. `RuntimeError: Expected all tensors to be on the same device`

**Příčina:** Model je na GPU, ale data na CPU (nebo naopak).

**Řešení:** Vždy přesuňte data i model na stejné zařízení:
```python
model = model.to(device)
X, y = X.to(device), y.to(device)
```

---

## 3. Model se neučí (loss neklesá)

**Možné příčiny:**
- Learning rate příliš vysoký → zkuste `lr=0.0001`
- Learning rate příliš nízký → zkuste `lr=0.01`
- Chybí `optimizer.zero_grad()` → gradienty se kumulují!
- Chybí `loss.backward()` → gradienty se nepočítají

---

## 4. `model.eval()` vs `model.train()`

**Problém:** Zapomínáte přepnout režim.

| Situace | Správný režim |
|---|---|
| Trénování | `model.train()` |
| Testování / predikce | `model.eval()` + `torch.no_grad()` |

---

## 5. `CUDA out of memory`

**Příčina:** GPU nemá dost paměti.

**Řešení:**
- Zmenšete `batch_size` (např. z 64 na 32)
- Používejte `torch.no_grad()` při inferenci
- Uvolněte paměť: `torch.cuda.empty_cache()`

---

## 6. Overfitting (přetrénování)

**Příznak:** Train loss klesá, ale test loss roste.

**Řešení:**
- Přidejte `nn.Dropout(0.5)` mezi vrstvy
- Použijte data augmentaci (otáčení, posun, zoom)
- Snižte počet epoch (early stopping)

---

## 7. Špatné rozměry obrázku pro predikci

**Příčina:** Model čeká tensor tvaru `(batch, channels, height, width)`, ale dostane jen `(channels, height, width)`.

**Řešení:** Přidejte batch dimenzi:
```python
img = img.unsqueeze(0)  # (1, 28, 28) → (1, 1, 28, 28)
```

---

## 8. Porovnání TensorFlow vs PyTorch

| TensorFlow/Keras | PyTorch | Poznámka |
|---|---|---|
| `model.fit(X, y, epochs=10)` | Ruční trénovací smyčka | PyTorch = více kontroly |
| `model.compile(loss=..., optimizer=...)` | `loss_fn = ...`, `optimizer = ...` | Oddělené objekty |
| `model.save('model.h5')` | `torch.save(model.state_dict(), 'model.pth')` | |
| `keras.Sequential([...])` | `nn.Sequential(...)` nebo `nn.Module` | |
| `Dense(128, activation='relu')` | `nn.Linear(in, 128)` + `nn.ReLU()` | Aktivace je zvlášť |

---

## 📚 Další zdroje

| Zdroj | Odkaz |
|---|---|
| Oficiální PyTorch tutoriály | https://docs.pytorch.org/tutorials/beginner/basics/intro.html |
| PyTorch dokumentace | https://docs.pytorch.org/docs/stable/index.html |
| 3Blue1Brown — neuronové sítě | https://www.youtube.com/watch?v=aircAruvnKk |
| fast.ai — praktický kurz | https://course.fast.ai/ |

---

**🎉 Gratulujeme!** Zvládli jste základy strojového učení s PyTorch. Umíte:
- ✅ Pracovat s tensory
- ✅ Načítat a připravovat data
- ✅ Stavět neuronové sítě
- ✅ Trénovat a vyhodnocovat modely
- ✅ Ukládat a načítat natrénované modely
- ✅ Predikovat z vlastních obrázků